# MVP - Engenharia de Dados
## Operação em solo de aeroportos e companhias aéreas: identificação de gargalos operacionais

**Fonte de dados:** ANAC — VRA (Voo Regular Ativo)
**Período analisado:** 2023–2025
**Escopo:** 15 maiores aeroportos do Brasil (como origem e/ou destino do voo)
**Plataforma:** Databricks Free Edition
**Arquitetura:** Medalhão (Bronze → Silver → Gold)

### Sobre o tema

A aviação civil brasileira opera sob uma tensão constante entre dois objetivos que nem sempre andam juntos: cumprir horários e manter a operação em solo enxuta o suficiente para ser rentável. Atrasos, cancelamentos e gargalos de turnaround não acontecem de forma aleatória — eles se concentram em determinados aeroportos, em determinadas companhias, em determinados períodos do ano, e enxergar esses padrões é o primeiro passo para qualquer decisão de investimento em infraestrutura aeroportuária ou de revisão de malha aérea.

A ANAC disponibiliza publicamente, através do programa VRA (Voo Regular Ativo), o registro de todos os voos regulares realizados no país, com horários previstos e realizados de partida e chegada, situação do voo (realizado/cancelado) e a justificativa reportada pela própria companhia aérea quando há atraso ou cancelamento. É uma base rica, mas bruta: nomes de aeroportos vêm por extenso junto com cidade e país, justificativas são texto livre sem padronização, e não há identificação de aeronave que permita calcular o tempo de giro em solo (turnaround) de forma exata — desafios que este pipeline precisa resolver antes de qualquer análise.

### Por que esse tema

Sou analista de negócios com foco em operações — hoje atuo no mercado/atacado do iFood, então lido diariamente com indicadores operacionais, gargalos de capacidade e trade-offs entre eficiência e nível de serviço, só que em um contexto completamente diferente (logística de entrega, não aviação). A aviação sempre foi um assunto que me interessa por conta própria, e esse MVP foi a oportunidade de aplicar o mesmo tipo de raciocínio analítico que uso no trabalho — identificar onde estão os gargalos e o que os está causando — em um domínio novo para mim, usando dados públicos e reais.

### Perguntas que o pipeline busca responder

1. Quais são as piores taxas de pontualidade por aeroporto e por companhia aérea?
    1. A pontualidade melhora ou piora ao longo dos meses analisados?
2. Quais são as justificativas de atraso mais frequentes reportadas pelas companhias?
3. Qual o turnaround (tempo de giro em solo) médio por aeroporto?
    1. Um turnaround médio alto se correlaciona com atraso médio de partida no mesmo aeroporto?
4. Quais aeroportos/companhias podem ser considerados referência de desempenho operacional, e quais são os principais candidatos a intervenção?
5. Existe sazonalidade na pontualidade (ex.: dezembro/janeiro e julho vs. demais meses)?
6. O gargalo observado é predominantemente do aeroporto ou da companhia aérea?

### Como este notebook está organizado

Segue a Arquitetura Medalhão: a Etapa 2 coleta e preserva os dados brutos sem qualquer alteração (Bronze); a Etapa 3 define o modelo de dados (fato + dimensões, com chaves primárias e estrangeiras); a Etapa 4 executa o ETL de tipagem, padronização e enriquecimento (Silver → Gold); a Etapa 5 verifica a qualidade e a integridade referencial dos dados antes de qualquer análise; e a Etapa 6 responde cada uma das perguntas acima com consultas SQL sobre a camada Gold.

Pode ser importado diretamente no Databricks (Workspace > Import > .ipynb) ou executado em qualquer ambiente Jupyter com uma SparkSession configurada (variável `spark`).

## Etapa 2 - Coleta de dados (camada Bronze)

**Importante:** o download direto dos CSVs a partir do notebook Databricks (via `requests`) não é
confiável na Free Edition, pois o compute serverless tem acesso restrito à internet externa.
Por isso, a coleta é feita em duas partes:

1. **Fora do Databricks** (na sua máquina local): rode o script `baixar_vra_anac.py`, que baixa
   todos os CSVs mensais do VRA diretamente do site da ANAC.
2. **Dentro do Databricks**: você faz o upload manual da pasta `VRA` (contendo todos os CSVs)
   para o Databricks — seja diretamente em um Volume (Catalog Explorer > seu volume > Upload),
   seja pela Workspace (file browser). O notebook abaixo localiza essa pasta automaticamente e
   copia os arquivos para o Volume, independente de onde ela foi enviada.

Nenhum tratamento é aplicado aos dados nesta camada - eles são preservados exatamente como
recebidos, garantindo rastreabilidade (princípio da camada Bronze).

In [0]:
catalog = "mvp_operacao_aerea"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"
volume_name = "vra_raw"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_bronze}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_silver}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_gold}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_bronze}.{volume_name}")

volume_path = f"/Volumes/{catalog}/{schema_bronze}/{volume_name}"
print(f"Volume bronze: {volume_path}")

### Localização e cópia dos CSVs da pasta "VRA" enviada
A célula abaixo procura uma pasta chamada `VRA` contendo os arquivos CSV, tanto dentro do
Volume quanto na Workspace pessoal (caso o upload tenha sido feito pelo file browser em vez do
Catalog Explorer), e copia todos os CSVs encontrados para o Volume — local de onde o restante
do pipeline lê os dados. O processo é registrado em um log, útil como evidência para o relatório,
e a célula falha com uma mensagem clara (em vez de um erro genérico de índice) caso a pasta não
seja encontrada em nenhuma das raízes candidatas.

In [0]:
import os
import glob
import shutil

nome_pasta = "VRA"
log = []

# possíveis raízes onde a pasta VRA pode ter sido enviada
usuario_atual = spark.sql("SELECT current_user()").collect()[0][0]
raizes_candidatas = [
    volume_path,                                   # upload direto dentro do Volume
    f"/Workspace/Users/{usuario_atual}",            # upload pelo file browser da Workspace
]

pastas_encontradas = []
for raiz in raizes_candidatas:
    pastas_encontradas += glob.glob(f"{raiz}/**/{nome_pasta}", recursive=True)
    # cobre também o caso em que a própria raiz é o volume e a pasta está um nível abaixo
    if os.path.basename(raiz.rstrip('/')) == nome_pasta:
        pastas_encontradas.append(raiz)

pastas_encontradas = sorted(set(p for p in pastas_encontradas if os.path.isdir(p)))

if not pastas_encontradas:
    mensagem = (
        f"Nenhuma pasta '{nome_pasta}' encontrada em nenhuma destas raízes:\n"
        + "\n".join(f"  - {r}" for r in raizes_candidatas)
        + "\nUse o ícone do arquivo no Databricks > 'Copy path' para confirmar o local exato "
        + "e ajuste `raizes_candidatas` se necessário."
    )
    raise FileNotFoundError(mensagem)

pasta_origem = pastas_encontradas[0]
log.append(f"Pasta encontrada em: {pasta_origem}")

csvs_origem = glob.glob(f"{pasta_origem}/*.csv")
log.append(f"CSVs encontrados na pasta: {len(csvs_origem)}")

copiados, ja_existiam = 0, 0
for caminho_csv in csvs_origem:
    nome_arquivo = os.path.basename(caminho_csv)
    destino = f"{volume_path}/{nome_arquivo}"
    if os.path.abspath(caminho_csv) == os.path.abspath(destino):
        ja_existiam += 1
        continue
    if os.path.exists(destino):
        ja_existiam += 1
        continue
    shutil.copy2(caminho_csv, destino)
    copiados += 1

log.append(f"Copiados para o volume agora: {copiados}")
log.append(f"Já estavam no volume: {ja_existiam}")

arquivos_finais = sorted(f for f in os.listdir(volume_path) if f.lower().endswith(".csv"))
log.append(f"Total de CSVs disponíveis no volume: {len(arquivos_finais)}")
log.append(f"Primeiro: {arquivos_finais[0] if arquivos_finais else 'N/A'}")
log.append(f"Último:   {arquivos_finais[-1] if arquivos_finais else 'N/A'}")

print("\n".join(log))

### Ingestão do bronze para tabela Delta (sem tratamento)
Os CSVs da ANAC usam separador `;`, encoding **ISO-8859-1 (latin1)** e decimal `,`. Aqui apenas
consolidamos os arquivos como estão — nenhuma limpeza é feita nesta etapa.

**Nota de revisão:** a primeira versão desta célula já documentava o encoding `latin1` aqui no
markdown, mas o parâmetro do leitor CSV estava configurado como `UTF-8` — um arquivo latin1 lido
como UTF-8 corrompe qualquer caractere acentuado (nomes de aeroporto, de companhia, etc.), o que
é crítico porque a modelagem da Etapa 3 passa a depender exatamente desses campos de texto.
Corrigido abaixo. Também adicionamos a coluna `codeshare` (que existe no CSV mas não estava no
mapeamento) e uma checagem defensiva caso o Volume esteja vazio.

In [0]:
from pyspark.sql import functions as F, Window
import unicodedata
import re

def sanitizar_nome_coluna(nome):
    # remove acentos, troca espaços/pontuação por underscore, tudo minúsculo
    sem_acento = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    limpo = re.sub(r"[^a-zA-Z0-9]+", "_", sem_acento).strip("_").lower()
    return limpo

# mapeamento dos nomes originais da ANAC (já sanitizados) para nomes curtos usados no pipeline
MAPA_COLUNAS = {
    "sigla_icao_empresa_aarea": "icao_empresa",
    "empresa_aarea": "nome_empresa_raw",
    "naomero_voo": "numero_voo",
    "ca3digo_di": "codigo_di",
    "ca3digo_tipo_linha": "codigo_tipo_linha",
    "modelo_equipamento": "modelo_equipamento",
    "naomero_de_assentos": "numero_assentos",
    "sigla_icao_aeroporto_origem": "icao_origem",
    "descriaao_aeroporto_origem": "descricao_aeroporto_origem",
    "partida_prevista": "partida_prevista_raw",
    "partida_real": "partida_real_raw",
    "sigla_icao_aeroporto_destino": "icao_destino",
    "descriaao_aeroporto_destino": "descricao_aeroporto_destino",
    "chegada_prevista": "chegada_prevista_raw",
    "chegada_real": "chegada_real_raw",
    "situaaao_voo": "situacao_voo",
    "justificativa": "justificativa",
    "referencia": "referencia_raw",
    "situaaao_partida": "situacao_partida",
    "situaaao_chegada": "situacao_chegada",
    "codeshare": "codeshare",
}

csvs_no_volume = [f for f in os.listdir(volume_path) if f.lower().endswith(".csv")]
if not csvs_no_volume:
    raise FileNotFoundError(
        f"Nenhum CSV encontrado em {volume_path} — rode a célula de localização/cópia acima antes desta."
    )

df_raw = (
    spark.read.format("csv")
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")  # ANAC publica os CSVs em latin1, não UTF-8
    .option("inferSchema", False)  # bronze: tudo como string, tipagem só na silver
    .load(f"{volume_path}/*.csv")
)

# sanitiza e renomeia colunas de forma dinâmica (resiliente a mudanças de layout entre anos)
colunas_sanitizadas = [sanitizar_nome_coluna(c) for c in df_raw.columns]
colunas_finais = [MAPA_COLUNAS.get(c, c) for c in colunas_sanitizadas]

df_bronze = (
    df_raw.toDF(*colunas_finais)
    .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
    .withColumn("_data_ingestao", F.current_timestamp())
)

(
    df_bronze.write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema_bronze}.vra_voos")
)

print(f"Colunas finais: {df_bronze.columns}")
print(f"Linhas na camada bronze: {df_bronze.count()}")
display(df_bronze.limit(5))

## Etapa 3 - Modelagem de dados

Modelo em **esquema flat por conceito** (Data Lake), composto por:
- `fVoos` (fato): uma linha por etapa de voo, filtrada para voos que tocam os 15 aeroportos do
  escopo (como origem OU destino), com os dois lados da rota enriquecidos (nome do aeroporto de
  origem **e** de destino) e a companhia identificada.
- `dAeroporto`: **todos** os aeroportos que aparecem em `icao_origem`/`icao_destino` na base (não
  só os 15 do escopo), com um indicador `aeroporto_monitorado` sinalizando quais deles fazem parte
  do escopo deste MVP.
- `dCia`: companhias aéreas identificadas na base, com nome comercial derivado da própria coluna
  `Empresa Aérea` do CSV.
- `dCalendario`: dimensão de datas (dia, mês, trimestre, indicador de alta temporada).

O catálogo de dados completo está no documento MVP_Operacao_Aerea.docx, seção 2.3.1.

**Nota de revisão:** a primeira versão deste notebook tinha `dAeroporto` como uma lista fixa de só
15 aeroportos e `dCia` como um dicionário manual de 9 companhias. Isso quebrava o modelo relacional:
qualquer voo cujo aeroporto do escopo fosse o *destino* (não a origem) ficava com o nome do
aeroporto nulo no Gold e desaparecia das análises agrupadas por aeroporto — e não havia nenhuma
chave primária/estrangeira declarada entre as tabelas. As duas dimensões foram reconstruídas para
cobrir 100% dos códigos referenciados por `fVoos`, e chaves primárias/estrangeiras foram
adicionadas (ver células abaixo).

### Dimensão de aeroportos (dAeroporto)

Construída a partir de **todos** os códigos ICAO que aparecem em `icao_origem` ou `icao_destino`
na camada Bronze — não apenas os 15 aeroportos do escopo. Para cada código, usamos a descrição
mais frequente (moda) entre `descricao_aeroporto_origem`/`descricao_aeroporto_destino` (a mesma
coluna que a ANAC já entrega no CSV), já que a mesma sigla pode aparecer com pequenas variações de
grafia entre arquivos mensais diferentes.

Para os 15 aeroportos do escopo, sobrepomos o nome/cidade curados manualmente (mais legíveis que o
texto bruto da ANAC); para os demais, o nome/cidade são derivados automaticamente da descrição
bruta (separando por ' - '). O indicador booleano `aeroporto_monitorado` sinaliza quais aeroportos
são os 15 do escopo — é esse indicador que as consultas da Etapa 6 usam para filtrar corretamente.

In [0]:
# candidatos de aeroporto observados na base (origem e destino), com a descrição já trazida pela ANAC
aeroportos_origem = df_bronze.select(
    F.col("icao_origem").alias("icao"),
    F.col("descricao_aeroporto_origem").alias("descricao_aeroporto"),
)
aeroportos_destino = df_bronze.select(
    F.col("icao_destino").alias("icao"),
    F.col("descricao_aeroporto_destino").alias("descricao_aeroporto"),
)
todos_aeroportos_raw = (
    aeroportos_origem.union(aeroportos_destino)
    .filter((F.col("icao").isNotNull()) & (F.col("icao") != ""))
    .filter((F.col("descricao_aeroporto").isNotNull()) & (F.col("descricao_aeroporto") != ""))
)

# a mesma sigla pode ter pequenas variações de grafia na descrição entre arquivos mensais;
# fica com a descrição mais frequente (moda) para cada código ICAO
contagem_descricao = todos_aeroportos_raw.groupBy("icao", "descricao_aeroporto").count()
w_moda = Window.partitionBy("icao").orderBy(F.desc("count"))
descricao_moda = (
    contagem_descricao.withColumn("rn", F.row_number().over(w_moda))
    .filter(F.col("rn") == 1)
    .select("icao", "descricao_aeroporto")
)

# separa "nome" e "cidade/UF ou país" a partir do padrão observado "NOME - CIDADE - UF/PAÍS"
partes = F.split(F.col("descricao_aeroporto"), " - ")
todos_aeroportos = descricao_moda.withColumn(
    "nome_aeroporto_derivado", partes.getItem(0)
).withColumn(
    "cidade_uf_derivado",
    F.when(F.size(partes) > 1, F.concat_ws(" - ", F.slice(partes, 2, 10))).otherwise(F.lit(None)),
)

# os 15 aeroportos do escopo deste MVP, com nome/cidade curados manualmente
aeroportos_escopo = [
    ("SBGR", "Guarulhos", "São Paulo/SP"),
    ("SBSP", "Congonhas", "São Paulo/SP"),
    ("SBBR", "Brasília", "Brasília/DF"),
    ("SBKP", "Viracopos", "Campinas/SP"),
    ("SBGL", "Galeão", "Rio de Janeiro/RJ"),
    ("SBCF", "Confins", "Belo Horizonte/MG"),
    ("SBRJ", "Santos Dumont", "Rio de Janeiro/RJ"),
    ("SBPA", "Salgado Filho", "Porto Alegre/RS"),
    ("SBRF", "Guararapes", "Recife/PE"),
    ("SBSV", "Dep. Luís Eduardo Magalhães", "Salvador/BA"),
    ("SBFZ", "Pinto Martins", "Fortaleza/CE"),
    ("SBCT", "Afonso Pena", "Curitiba/PR"),
    ("SBBE", "Val de Cans", "Belém/PA"),
    ("SBEG", "Eduardo Gomes", "Manaus/AM"),
    ("SBFL", "Hercílio Luz", "Florianópolis/SC"),
]
dAeroporto_curado = spark.createDataFrame(
    aeroportos_escopo, schema=["icao", "nome_aeroporto_curado", "cidade_uf_curado"]
)
codigos_escopo = [a[0] for a in aeroportos_escopo]

dAeroporto = (
    todos_aeroportos.join(dAeroporto_curado, "icao", "left")
    .withColumn("nome_aeroporto", F.coalesce(F.col("nome_aeroporto_curado"), F.col("nome_aeroporto_derivado")))
    .withColumn("cidade_uf", F.coalesce(F.col("cidade_uf_curado"), F.col("cidade_uf_derivado")))
    .withColumn("aeroporto_monitorado", F.col("icao").isin(codigos_escopo))
    .select("icao", "nome_aeroporto", "cidade_uf", "aeroporto_monitorado", "descricao_aeroporto")
)

dAeroporto.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_gold}.dAeroporto"
)
print(f"Total de aeroportos distintos na base: {dAeroporto.count()}")
print(f"Aeroportos monitorados (escopo do MVP): {dAeroporto.filter('aeroporto_monitorado').count()} de {len(aeroportos_escopo)} esperados")
display(dAeroporto.filter("aeroporto_monitorado").orderBy("icao"))

### Dimensão de companhias aéreas (dCia)

Construída a partir do nome mais frequente (moda) de `nome_empresa_raw` — a coluna `Empresa Aérea`
do CSV original — para cada `icao_empresa` observado na Bronze. Isso garante cobertura de 100% das
companhias presentes na base, diferente de um dicionário manual fixo.

Um pequeno dicionário de *override* (`mapa_cias_override`) é aplicado por cima só para os casos em
que o nome bruto da ANAC vem inconsistente entre arquivos (ex.: variações de razão social ao longo
do tempo) — ele tem prioridade sobre o nome derivado dos dados, mas deixou de ser a fonte primária.

In [0]:
# nome mais frequente (moda) de cada companhia, a partir da própria coluna "Empresa Aérea" do CSV
contagem_nome_cia = (
    df_bronze.filter((F.col("icao_empresa").isNotNull()) & (F.col("icao_empresa") != ""))
    .groupBy("icao_empresa", "nome_empresa_raw")
    .count()
)
w_moda_cia = Window.partitionBy("icao_empresa").orderBy(F.desc("count"))
nome_cia_moda = (
    contagem_nome_cia.withColumn("rn", F.row_number().over(w_moda_cia))
    .filter(F.col("rn") == 1)
    .select(F.col("icao_empresa").alias("icao"), F.col("nome_empresa_raw").alias("nome_cia_dados"))
)

# override manual só para os casos em que o nome bruto vem inconsistente entre arquivos
# (deixou de ser a fonte primária do nome; agora é só uma exceção documentada)
mapa_cias_override = {
    "AZU": "Azul",
    "GLO": "GOL",
    "TAM": "LATAM Airlines Brasil",
    "PTB": "Passaredo",
    "ONE": "Avianca Brasil",
    "TTL": "Total Linhas Aéreas",
    "SID": "Sideral Linhas Aéreas",
    "MWM": "MAP Linhas Aéreas",
    "ABJ": "ABSA Cargo",
}
mapa_override_df = spark.createDataFrame(
    list(mapa_cias_override.items()), schema=["icao", "nome_cia_override"]
)

dCia = (
    nome_cia_moda.join(mapa_override_df, "icao", "left")
    .withColumn("nome_cia", F.coalesce(F.col("nome_cia_override"), F.col("nome_cia_dados")))
    .select("icao", "nome_cia")
)
dCia.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_gold}.dCia"
)
print(f"Companhias identificadas na base: {dCia.count()}")
display(dCia.orderBy("icao"))

### Dimensão de calendário (dCalendario)
Completa o modelo: dimensão de datas cobrindo todo o período dos dados (2023-01-01 a 2025-12-31),
com granularidade de dia/mês/trimestre e um indicador de `alta_temporada` (dezembro, janeiro e
julho), usado na Pergunta 5 (sazonalidade).


In [0]:
from datetime import date, timedelta
import calendar as cal_mod

data_inicio = date(2023, 1, 1)
data_fim = date(2025, 12, 31)

linhas_calendario = []
d = data_inicio
while d <= data_fim:
    linhas_calendario.append({
        "date": d,
        "day_of_month": d.day,
        "day_name": d.strftime("%A"),
        "month": d.month,
        "month_name": d.strftime("%B"),
        "quarter": (d.month - 1) // 3 + 1,
        "year": d.year,
        "days_in_month": cal_mod.monthrange(d.year, d.month)[1],
        "alta_temporada": d.month in (12, 1, 7),
    })
    d += timedelta(days=1)

dCalendario = spark.createDataFrame(linhas_calendario)
dCalendario.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_gold}.dCalendario"
)
print(f"Linhas em dCalendario: {dCalendario.count()}")
display(dCalendario.limit(5))

### Chaves primárias das dimensões

Até aqui, o relacionamento entre `fVoos`, `dAeroporto`, `dCia` e `dCalendario` existia só
implicitamente, dentro dos `.join()` do PySpark — não havia nenhuma chave primária/estrangeira
declarada no Unity Catalog, então o Catalog Explorer do Databricks não desenhava nenhum
relacionamento entre as tabelas.

A célula abaixo declara `PRIMARY KEY` nas três dimensões. Duas observações:
- No Unity Catalog, chave primária exige que a coluna seja `NOT NULL` — por isso ajustamos isso
  antes de declarar a chave.
- Constraints de `PRIMARY KEY`/`FOREIGN KEY` no Delta/Unity Catalog são **informativas** (não são
  impostas pelo motor de escrita): documentam o modelo e habilitam recursos como o diagrama de
  relacionamento do Catalog Explorer, mas não bloqueiam uma escrita que viole a chave. A garantia
  de integridade real vem de como o pipeline foi construído (as duas dimensões cobrem 100% dos
  códigos referenciados por `fVoos`, por construção) — conferido na Etapa 5.

As chaves de `fVoos` e de `turnaround_estimado` (primárias e estrangeiras, incluindo para
`dCalendario`) são declaradas mais adiante, na Etapa 4, logo depois que cada uma dessas tabelas é
criada (não é possível referenciar ou colocar PRIMARY KEY numa tabela que ainda não existe).

In [0]:
def executar_ddl(comandos, titulo):
    print(f"--- {titulo} ---")
    for comando in comandos:
        try:
            spark.sql(comando)
            print(f"OK: {comando}")
        except Exception as erro:
            print(f"AVISO (ignorado — provável constraint já existente ou runtime sem suporte): {comando}\n  -> {erro}")

ddl_not_null_dims = [
    f"ALTER TABLE {catalog}.{schema_gold}.dAeroporto ALTER COLUMN icao SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema_gold}.dCia ALTER COLUMN icao SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema_gold}.dCalendario ALTER COLUMN date SET NOT NULL",
]
executar_ddl(ddl_not_null_dims, "Colunas de chave como NOT NULL")

ddl_pk = [
    f"ALTER TABLE {catalog}.{schema_gold}.dAeroporto DROP CONSTRAINT IF EXISTS pk_daeroporto",
    f"ALTER TABLE {catalog}.{schema_gold}.dAeroporto ADD CONSTRAINT pk_daeroporto PRIMARY KEY (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.dCia DROP CONSTRAINT IF EXISTS pk_dcia",
    f"ALTER TABLE {catalog}.{schema_gold}.dCia ADD CONSTRAINT pk_dcia PRIMARY KEY (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.dCalendario DROP CONSTRAINT IF EXISTS pk_dcalendario",
    f"ALTER TABLE {catalog}.{schema_gold}.dCalendario ADD CONSTRAINT pk_dcalendario PRIMARY KEY (date)",
]
executar_ddl(ddl_pk, "Chaves primárias das dimensões")

## Etapa 4 - Carga (ETL): Bronze -> Silver -> Gold

In [0]:
df_silver = (
    spark.table(f"{catalog}.{schema_bronze}.vra_voos")
    .withColumn("partida_prevista", F.to_timestamp("partida_prevista_raw", "dd/MM/yyyy HH:mm"))
    .withColumn("partida_real", F.to_timestamp("partida_real_raw", "dd/MM/yyyy HH:mm"))
    .withColumn("chegada_prevista", F.to_timestamp("chegada_prevista_raw", "dd/MM/yyyy HH:mm"))
    .withColumn("chegada_real", F.to_timestamp("chegada_real_raw", "dd/MM/yyyy HH:mm"))
    .withColumn("numero_assentos", F.col("numero_assentos").cast("int"))
    .withColumn(
        "atraso_partida_min",
        (F.col("partida_real").cast("long") - F.col("partida_prevista").cast("long")) / 60,
    )
    .withColumn(
        "atraso_chegada_min",
        (F.col("chegada_real").cast("long") - F.col("chegada_prevista").cast("long")) / 60,
    )
    # escopo: mantém a etapa se origem OU destino estiver entre os 15 aeroportos monitorados
    .filter(
        F.col("icao_origem").isin(codigos_escopo)
        | F.col("icao_destino").isin(codigos_escopo)
    )
    # a partir daqui só seguem colunas de negócio já tipadas — os campos _raw só existiam para
    # alimentar o to_timestamp acima, e os metadados de controle (_arquivo_origem, _data_ingestao)
    # são responsabilidade da Bronze, que já preserva rastreabilidade completa. Nenhum dos dois
    # precisa (nem deveria) seguir até a Silver/Gold.
    .select(
        "icao_empresa", "numero_voo", "codigo_di", "codigo_tipo_linha", "modelo_equipamento",
        "numero_assentos", "icao_origem", "icao_destino",
        "partida_prevista", "partida_real", "chegada_prevista", "chegada_real",
        "atraso_partida_min", "atraso_chegada_min",
        "situacao_voo", "justificativa", "situacao_partida", "situacao_chegada", "codeshare",
    )
)

df_silver.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_silver}.vra_voos"
)
print(f"Linhas na camada silver (filtradas ao escopo): {df_silver.count()}")

### Turnaround (proxy)
A base VRA não identifica a aeronave (sem número de matrícula), portanto o turnaround real
por aeronave não pode ser calculado com precisão. Como **proxy**, estimamos o intervalo entre
uma chegada e a próxima partida da **mesma companhia no mesmo aeroporto no mesmo dia**.

In [0]:
chegadas = df_silver.filter(F.col("chegada_real").isNotNull()).select(
    F.col("icao_destino").alias("aeroporto"),
    F.col("icao_empresa"),
    F.col("chegada_real").alias("evento_hora"),
    F.lit("chegada").alias("tipo_evento"),
)
partidas = df_silver.filter(F.col("partida_real").isNotNull()).select(
    F.col("icao_origem").alias("aeroporto"),
    F.col("icao_empresa"),
    F.col("partida_real").alias("evento_hora"),
    F.lit("partida").alias("tipo_evento"),
)
eventos = chegadas.union(partidas)

w = Window.partitionBy("aeroporto", "icao_empresa").orderBy("evento_hora")
eventos = eventos.withColumn("proximo_evento", F.lead("tipo_evento").over(w)).withColumn(
    "proxima_hora", F.lead("evento_hora").over(w)
)

turnaround = (
    eventos.filter((F.col("tipo_evento") == "chegada") & (F.col("proximo_evento") == "partida"))
    .withColumn(
        "turnaround_min",
        (F.col("proxima_hora").cast("long") - F.col("evento_hora").cast("long")) / 60,
    )
    .filter((F.col("turnaround_min") > 0) & (F.col("turnaround_min") < 720))  # remove outliers (>12h = não é turnaround real)
    .withColumn("data_evento", F.to_date("evento_hora"))
    .select("aeroporto", "icao_empresa", "evento_hora", "data_evento", "turnaround_min")
)

turnaround.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_gold}.turnaround_estimado"
)
print(f"Registros de turnaround estimado: {turnaround.count()}")

### Chaves de turnaround_estimado

Diferente das dimensões, aqui não existe uma coluna única e óbvia como identificador — a chave
natural é a combinação `(aeroporto, icao_empresa, evento_hora)`: "esta chegada específica, desta
companhia, neste aeroporto". Ela só quebra se duas aeronaves da mesma companhia pousarem no mesmo
aeroporto no mesmo minuto, o que não é impossível num hub movimentado — por isso a contagem de
duplicatas dessa combinação é conferida explicitamente na Etapa 5, junto com a integridade
referencial, em vez de simplesmente presumida.

Também é adicionada a coluna `data_evento` (a data, sem hora, da chegada que inicia o turnaround),
usada só para permitir a chave estrangeira com `dCalendario` — nenhuma das 8 perguntas atuais
cruza turnaround com sazonalidade, mas o modelo fica pronto para isso.

In [0]:
ddl_not_null_turnaround = [
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ALTER COLUMN aeroporto SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ALTER COLUMN icao_empresa SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ALTER COLUMN evento_hora SET NOT NULL",
]
executar_ddl(ddl_not_null_turnaround, "turnaround_estimado: colunas de chave como NOT NULL")

ddl_pk_turnaround = [
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado DROP CONSTRAINT IF EXISTS pk_turnaround_estimado",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ADD CONSTRAINT pk_turnaround_estimado PRIMARY KEY (aeroporto, icao_empresa, evento_hora)",
]
executar_ddl(ddl_pk_turnaround, "turnaround_estimado: chave primária composta")

ddl_fk_turnaround = [
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado DROP CONSTRAINT IF EXISTS fk_turnaround_daeroporto",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ADD CONSTRAINT fk_turnaround_daeroporto FOREIGN KEY (aeroporto) REFERENCES {catalog}.{schema_gold}.dAeroporto (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado DROP CONSTRAINT IF EXISTS fk_turnaround_dcia",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ADD CONSTRAINT fk_turnaround_dcia FOREIGN KEY (icao_empresa) REFERENCES {catalog}.{schema_gold}.dCia (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado DROP CONSTRAINT IF EXISTS fk_turnaround_dcalendario",
    f"ALTER TABLE {catalog}.{schema_gold}.turnaround_estimado ADD CONSTRAINT fk_turnaround_dcalendario FOREIGN KEY (data_evento) REFERENCES {catalog}.{schema_gold}.dCalendario (date)",
]
executar_ddl(ddl_fk_turnaround, "turnaround_estimado: chaves estrangeiras")

### Tabela Gold consolidada (fVoos)

Junta a camada Silver com a dimensão de aeroporto **duas vezes** (uma para `icao_origem`, outra
para `icao_destino`) e com a dimensão de companhia, pronta para consulta em SQL.

Diferente da primeira versão (que só enriquecia o lado da origem), agora `fVoos` traz
`nome_aeroporto_origem` **e** `nome_aeroporto_destino`, além dos indicadores booleanos
`origem_no_escopo`/`destino_no_escopo` — usados nas consultas da Etapa 6 para deixar explícito
qual "lado" de cada pergunta de negócio está sendo medido (ex.: pontualidade de partida por
aeroporto só faz sentido filtrando `origem_no_escopo`).

Também adiciona `data_partida_prevista` (a data, sem hora, da partida prevista), usada como chave
estrangeira para `dCalendario` — é o mesmo join que a Pergunta 5 já fazia via `DATE(partida_prevista)`,
só que agora materializado como coluna em vez de recalculado a cada consulta.

In [0]:
# Quarentena: partida_prevista nula não é um estado de negócio válido (diferente de partida_real/
# chegada_real nulos, que representam cancelamento) — é um registro incompleto na fonte, e viola a
# PRIMARY KEY de fVoos (icao_empresa, numero_voo, partida_prevista), que exige NOT NULL. Por isso
# essas linhas são separadas para uma tabela de rejeitados em vez de entrar na Gold ou serem
# simplesmente descartadas sem registro.
fVoos_rejeitados = df_silver.filter(
    F.col("partida_prevista").isNull() |
    ((F.col("chegada_real").isNull()) & (F.col("situacao_voo") == "REALIZADO")) |
    ((F.col("partida_real").isNull()) & (F.col("situacao_voo") == "REALIZADO"))
)
fVoos_base = df_silver.filter(
    F.col("partida_prevista").isNotNull() &
    ~(((F.col("chegada_real").isNull()) & (F.col("situacao_voo") == "REALIZADO")) |
      ((F.col("partida_real").isNull()) & (F.col("situacao_voo") == "REALIZADO")))
)

qtd_rejeitados = fVoos_rejeitados.count()
qtd_total = df_silver.count()
print(f"Registros sem partida_prevista, movidos para fVoos_rejeitados: {qtd_rejeitados} de {qtd_total} ({100*qtd_rejeitados/qtd_total:.2f}%)")

fVoos_rejeitados.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_gold}.fVoos_rejeitados"
)

dAeroporto_origem = dAeroporto.select(
    F.col("icao").alias("icao_origem_lookup"),
    F.col("nome_aeroporto").alias("nome_aeroporto_origem"),
    F.col("cidade_uf").alias("cidade_uf_origem"),
    F.col("aeroporto_monitorado").alias("origem_no_escopo"),
)
dAeroporto_destino = dAeroporto.select(
    F.col("icao").alias("icao_destino_lookup"),
    F.col("nome_aeroporto").alias("nome_aeroporto_destino"),
    F.col("cidade_uf").alias("cidade_uf_destino"),
    F.col("aeroporto_monitorado").alias("destino_no_escopo"),
)

fVoos = (
    fVoos_base
    .join(dAeroporto_origem, fVoos_base.icao_origem == F.col("icao_origem_lookup"), "left")
    .drop("icao_origem_lookup")
    .join(dAeroporto_destino, fVoos_base.icao_destino == F.col("icao_destino_lookup"), "left")
    .drop("icao_destino_lookup")
    .join(dCia, fVoos_base.icao_empresa == dCia.icao, "left")
    .drop(dCia.icao)
    .withColumn("data_partida_prevista", F.to_date("partida_prevista"))
)

fVoos.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema_gold}.fVoos"
)
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema_gold}")
print("Tabela fVoos (gold) pronta para consultas em SQL.")

### Chaves primárias e estrangeiras de fVoos

`fVoos` não tem uma coluna única óbvia como identificador. A candidata natural de chave composta é
`(icao_empresa, numero_voo, partida_prevista)` — companhia + número do voo + horário previsto de
partida deveria identificar uma etapa de voo específica. Não encontrei confirmação oficial da ANAC
de que `codigo_di` sozinho seja garantidamente único por etapa, então não declaro PK em cima dele
sem verificar; a composta é a opção defensável, e sua unicidade é conferida na Etapa 5 (mesmo
raciocínio do `turnaround_estimado`).

Quanto às chaves estrangeiras: `fk_fvoos_dcia` é garantida por construção (`dCia` cobre 100% dos
códigos de companhia vistos na Bronze). `fk_fvoos_daeroporto_origem`/`_destino` só passam a ser
válidas depois da correção da Etapa 3 — antes, com `dAeroporto` limitada aos 15 aeroportos do
escopo, boa parte dos códigos de `icao_origem`/`icao_destino` de `fVoos` (aeroportos fora do
escopo, do outro lado da rota) não existia na dimensão. `fk_fvoos_dcalendario` usa a coluna
`data_partida_prevista` criada acima, já que uma FK precisa comparar colunas do mesmo tipo
(`date`), não uma expressão como `DATE(partida_prevista)` contra `dCalendario.date`.

In [0]:
ddl_not_null_fvoos = [
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ALTER COLUMN icao_empresa SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ALTER COLUMN numero_voo SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ALTER COLUMN partida_prevista SET NOT NULL",
]
executar_ddl(ddl_not_null_fvoos, "fVoos: colunas de chave como NOT NULL")

ddl_pk_fvoos = [
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos DROP CONSTRAINT IF EXISTS pk_fvoos",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ADD CONSTRAINT pk_fvoos PRIMARY KEY (icao_empresa, numero_voo, partida_prevista)",
]
executar_ddl(ddl_pk_fvoos, "fVoos: chave primária composta")

ddl_fk = [
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos DROP CONSTRAINT IF EXISTS fk_fvoos_dcia",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ADD CONSTRAINT fk_fvoos_dcia FOREIGN KEY (icao_empresa) REFERENCES {catalog}.{schema_gold}.dCia (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos DROP CONSTRAINT IF EXISTS fk_fvoos_daeroporto_origem",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ADD CONSTRAINT fk_fvoos_daeroporto_origem FOREIGN KEY (icao_origem) REFERENCES {catalog}.{schema_gold}.dAeroporto (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos DROP CONSTRAINT IF EXISTS fk_fvoos_daeroporto_destino",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ADD CONSTRAINT fk_fvoos_daeroporto_destino FOREIGN KEY (icao_destino) REFERENCES {catalog}.{schema_gold}.dAeroporto (icao)",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos DROP CONSTRAINT IF EXISTS fk_fvoos_dcalendario",
    f"ALTER TABLE {catalog}.{schema_gold}.fVoos ADD CONSTRAINT fk_fvoos_dcalendario FOREIGN KEY (data_partida_prevista) REFERENCES {catalog}.{schema_gold}.dCalendario (date)",
]
executar_ddl(ddl_fk, "Chaves estrangeiras de fVoos para as dimensões")

## Etapa 5 - Qualidade de dados
Verificação de nulos, domínios e consistência por atributo.

In [0]:
total = fVoos.count()
qualidade = []
for c in ["icao_empresa", "icao_origem", "icao_destino", "situacao_voo", "partida_real", "chegada_real"]:
    nulos = fVoos.filter(F.col(c).isNull()).count()
    qualidade.append((c, total, nulos, round(100 * nulos / total, 2)))

df_qualidade = spark.createDataFrame(
    qualidade, schema=["coluna", "total_linhas", "qtd_nulos", "pct_nulos"]
)
display(df_qualidade)

# domínio esperado de situacao_voo
print("Valores distintos de situacao_voo:")
fVoos.select("situacao_voo").distinct().show()

# checagem: atrasos negativos extremos (>1440 min = provável erro de digitação de data)
print("Registros com atraso de partida > 24h (suspeitos):")
print(fVoos.filter(F.abs("atraso_partida_min") > 1440).count())

**Interpretação esperada:** nulos em `partida_real`/`chegada_real` são esperados para voos
cancelados (situacao_voo = 'CANCELADO') e não devem ser removidos — apenas tratados como
ausência de execução na análise de pontualidade. Nulos em outros campos indicam problema
de qualidade a ser investigado.

### Integridade referencial

As chaves primárias e estrangeiras declaradas nas Etapas 3/4 não são impostas pelo Delta
(constraints informativas — ver nota na Etapa 3). Para ter certeza de que o modelo é de fato
íntegro (e não só documentado como se fosse), a célula abaixo mede diretamente eventuais
problemas: voos sem nenhum aeroporto no escopo, nomes de aeroporto/companhia não encontrados nas
dimensões, e duplicatas nas chaves primárias compostas de `fVoos` e `turnaround_estimado`
(declaradas na Etapa 4, mas nunca verificadas até agora). Todos os contadores abaixo devem ser 0.

In [0]:
print("=== Checagem de integridade referencial (valida o modelo/PK/FK das Etapas 3-4) ===\n")

sem_lado_no_escopo = fVoos.filter(~F.col("origem_no_escopo") & ~F.col("destino_no_escopo")).count()
print(f"Voos sem NENHUM lado (origem/destino) no escopo monitorado (esperado: 0, é a checagem do filtro da Silver): {sem_lado_no_escopo}")

sem_nome_origem = fVoos.filter(F.col("nome_aeroporto_origem").isNull()).count()
sem_nome_destino = fVoos.filter(F.col("nome_aeroporto_destino").isNull()).count()
print(f"Voos com nome_aeroporto_origem nulo (cobertura da nova dAeroporto, esperado: 0): {sem_nome_origem}")
print(f"Voos com nome_aeroporto_destino nulo (cobertura da nova dAeroporto, esperado: 0): {sem_nome_destino}")

sem_nome_cia = fVoos.filter(F.col("nome_cia").isNull()).count()
print(f"Voos com nome_cia nulo (cobertura da nova dCia, esperado: 0): {sem_nome_cia}")

sem_data_partida = fVoos.filter(F.col("data_partida_prevista").isNull()).count()
print(f"Voos com data_partida_prevista nula (partida_prevista não parseou, esperado: 0): {sem_data_partida}")

print()
dup_fvoos = (
    fVoos.groupBy("icao_empresa", "numero_voo", "partida_prevista")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"fVoos: combinações duplicadas da PK (icao_empresa, numero_voo, partida_prevista) — esperado 0: {dup_fvoos}")

dup_turnaround = (
    turnaround.groupBy("aeroporto", "icao_empresa", "evento_hora")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"turnaround_estimado: combinações duplicadas da PK (aeroporto, icao_empresa, evento_hora) — esperado 0: {dup_turnaround}")

**Se algum contador de duplicatas vier maior que 0:** a chave primária composta não é válida
como declarada para os seus dados. O mais provável, no caso de `fVoos`, é a presença de voos
codeshare (a coluna `codeshare`, disponível mas ainda não explorada, é o primeiro lugar a
investigar); no caso de `turnaround_estimado`, seria duas aeronaves da mesma companhia chegando no
mesmo aeroporto no mesmo minuto. De qualquer forma, o `ALTER TABLE ... ADD CONSTRAINT` não falha
nesse cenário (constraint informativa), então isso não quebra a execução — mas deve ser registrado
na Autoavaliação como uma limitação conhecida caso ocorra.

## Etapa 6 - Solução do problema proposto
Consultas SQL para cada pergunta de negócio definida no objetivo.

### Pergunta 1: piores taxas de pontualidade por aeroporto e companhia

In [0]:
display(spark.sql("""SELECT
  nome_aeroporto_origem,
  nome_cia,
  COUNT(*) AS total_voos,
  ROUND(100.0 * SUM(CASE WHEN atraso_partida_min <= 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_pontualidade
FROM fVoos
WHERE situacao_voo = 'REALIZADO' AND origem_no_escopo
GROUP BY nome_aeroporto_origem, nome_cia
HAVING total_voos > 100
ORDER BY pct_pontualidade ASC"""))

**Interpretação:** O topo da tabela encontramos apenas linhas aéreas cargueiras, o que é interessante de analisar pensando que quando se trata de cargas, 15 minutos de atraso não é um tempo tão relevante assim. Porém mais no meio da tabela vemos que é bem distribuido, não parecendo um problema por companhia pois as mesmas companhias que estão mais no meio da tabela tem destaque também no bottom da tabela. Exemplo a azul, em guarulhos, tem 75% de pontualidade. Mas em outros aeroportos possuem mais que 80%, num contexto geral. Gerando a duvida: Será que em grandes companhias como Gol, azul e Latam, o gargalo está mais relacionado ao aeroporto e não a companhia?

### Pergunta 1.a: evolução mensal da pontualidade (melhorando ou piorando?)

In [0]:
display(spark.sql("""SELECT
  date_trunc('month', partida_prevista) AS mes,
  ROUND(100.0 * SUM(CASE WHEN atraso_partida_min <= 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_pontualidade
FROM fVoos
WHERE situacao_voo = 'REALIZADO' AND origem_no_escopo
GROUP BY mes
ORDER BY mes"""))

In [0]:
%sql
select count(*),date_trunc('month', partida_prevista) AS mes from fVoos group by mes order by mes

**Interpretação:** Aqui não parece ter uma evolução clara, porém ao rodar a primeira query vemos claramente um outlier em maio de 2023. Ao rodar uma segunda query percebemos que estamos praticamente sem dados em maio de 2023, mostrando um erro na tabela. Investigando as origens vemos que o erro está em um dos CSV's de entrada.

### Pergunta 2: justificativas de atraso mais frequentes
-- Observação: 'justificativa' é texto livre preenchido pela própria companhia aérea,
-- não um código padronizado. Textos muito parecidos podem estar grafados de forma diferente
-- (ex.: variações de maiúscula/minúscula ou pontuação) e merecer agrupamento manual.

**Observação de escopo:** aqui, propositalmente, não filtramos por `origem_no_escopo` — a
justificativa de atraso é reportada pela companhia independentemente de qual lado (origem/destino)
é o aeroporto monitorado, então olhar para todo o `fVoos` dá uma leitura mais completa das causas
de atraso que tocam a operação desses aeroportos.

In [0]:
display(spark.sql("""SELECT
  TRIM(UPPER(justificativa)) AS justificativa_normalizada,
  COUNT(*) AS ocorrencias,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_do_total
FROM fVoos
WHERE justificativa IS NOT NULL AND TRIM(justificativa) <> ''
GROUP BY justificativa_normalizada
ORDER BY ocorrencias DESC
LIMIT 30"""))

**Interpretação:** Curiosamente as justificativas de atraso não estão preenchidas, verificando os csv's de entrada essa hipótese se confirma. As companhias não tem o costume de preencher essa informação ou o sistema que exportou os csvs para o site da ANAC não exporta essa informação.

### Pergunta 3: turnaround médio por aeroporto

Como `dAeroporto` agora cobre todos os códigos ICAO observados na base (não só os 15 do escopo),
filtramos explicitamente por `aeroporto_monitorado` — sem esse filtro, o turnaround seria
calculado também para aeroportos fora do escopo, com cobertura de tráfego parcial (só a fatia que
conecta com nossos 15 aeroportos) e sem significado estatístico.

In [0]:
display(spark.sql("""SELECT
  a.nome_aeroporto,
  ROUND(AVG(t.turnaround_min), 1) AS turnaround_medio_min,
  COUNT(*) AS qtd_amostras
FROM turnaround_estimado t
JOIN dAeroporto a ON t.aeroporto = a.icao
WHERE a.aeroporto_monitorado
GROUP BY a.nome_aeroporto
ORDER BY turnaround_medio_min DESC"""))

**Interpretação:** Observando os aeroportos com maiores turnaround vemos que Guarulhos, Eduardo Gomes e Galeão possuem valores muito altos, gerando um maior custo para a companhia aérea, visto que o avião no solo gera custo enquanto um avião no ar gera lucro. Assim percebemos também que Congonhas possui uma operação em solo exemplar, em questão de turnaround.

### Pergunta 3.a: turnaround alto correlaciona com atraso médio de partida no mesmo aeroporto?

Também passa a mostrar o nome do aeroporto (a versão anterior exibia só o código ICAO, pois a
query não fazia join com `dAeroporto`), e restringe a `aeroporto_monitorado` pelo mesmo motivo da
Pergunta 3.

In [0]:
display(spark.sql("""WITH turnaround_aeroporto AS (
  SELECT aeroporto, AVG(turnaround_min) AS turnaround_medio
  FROM turnaround_estimado
  GROUP BY aeroporto
),
atraso_aeroporto AS (
  SELECT icao_origem AS aeroporto, AVG(atraso_partida_min) AS atraso_medio
  FROM fVoos
  WHERE situacao_voo = 'REALIZADO' AND origem_no_escopo
  GROUP BY icao_origem
)
SELECT
  a.nome_aeroporto,
  ta.turnaround_medio,
  aa.atraso_medio
FROM turnaround_aeroporto ta
JOIN atraso_aeroporto aa ON ta.aeroporto = aa.aeroporto
JOIN dAeroporto a ON ta.aeroporto = a.icao
WHERE a.aeroporto_monitorado
ORDER BY ta.turnaround_medio DESC"""))

**Interpretação:** Não existe uma correlação direta, mas os 2 aeroportos com maior turnaround tem também um alto atraso médio. Mas os valores intermediários oscilam, não confirmando essa hipótese. Além disso, Congonhas possui o menor turnaround e não possui o menor atraso médio

### Pergunta 4: aeroportos/companhias referência vs. candidatos a intervenção

In [0]:
display(spark.sql("""SELECT
  nome_aeroporto_origem,
  ROUND(100.0 * SUM(CASE WHEN atraso_partida_min <= 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_pontualidade,
  ROUND(100.0 * SUM(CASE WHEN situacao_voo = 'CANCELADO' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_cancelamento
FROM fVoos
WHERE origem_no_escopo
GROUP BY nome_aeroporto_origem
ORDER BY pct_pontualidade DESC"""))

**Interpretação:** Brasilia lidera em pontualidade e percentual de cancelamento, sendo um otimo candidato a aeroporto de referência em operação. Já como candidato a intervenção, temos eduardo gomes com um alto percentual de cancelamento e baixa pontualidade

### Pergunta 5: sazonalidade (dezembro/janeiro e julho vs. demais meses)
Usa a chave estrangeira `fVoos.data_partida_prevista → dCalendario.date` (declarada na Etapa 4)
para trazer o indicador `alta_temporada` já calculado, em vez de extrair o mês diretamente do
timestamp ou recalcular `DATE(partida_prevista)` a cada consulta.

**Observação de escopo:** diferente das Perguntas 1, 1.a, 4 e 6, aqui não restringimos a
`origem_no_escopo` — a sazonalidade é medida sobre todo o movimento que toca os aeroportos do
escopo (partindo ou chegando), não apenas sobre os voos que partem deles.

In [0]:
display(spark.sql("""SELECT
  c.month AS mes_do_ano,
  c.month_name AS mes,
  c.alta_temporada,
  ROUND(100.0 * SUM(CASE WHEN f.atraso_partida_min <= 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_pontualidade,
  ROUND(AVG(f.atraso_partida_min), 1) AS atraso_medio_min,
  COUNT(*) AS total_voos
FROM fVoos f
JOIN dCalendario c ON f.data_partida_prevista = c.date
WHERE f.situacao_voo = 'REALIZADO'
GROUP BY c.month, c.month_name, c.alta_temporada
ORDER BY c.month"""))

**Interpretação:** Não parece existir uma piora por sazonalidade, mas existe sim uma piora no serviço no ultimo trimestre. Mas a hipótese é anulada quando julho, um mês de alta temporada, possui uma das maiores performances operacionais

### Pergunta 6: gargalo é do aeroporto ou da companhia?
-- (dispersão de desempenho entre companhias no mesmo aeroporto)

In [0]:
display(spark.sql("""SELECT
  nome_aeroporto_origem,
  nome_cia,
  ROUND(100.0 * SUM(CASE WHEN atraso_partida_min <= 15 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_pontualidade,
  COUNT(*) AS total_voos
FROM fVoos
WHERE situacao_voo = 'REALIZADO' AND origem_no_escopo
GROUP BY nome_aeroporto_origem, nome_cia
HAVING total_voos > 200
ORDER BY nome_aeroporto_origem, pct_pontualidade DESC"""))

**Interpretação:** A resposta é bem simples analisando os dados acima> É a companhia e não o aeroporto.

A evidência mais forte é o aeroporto de viracopos, onde as cargueiras variam performance de 90% de pontualidade (FedEx) vs 14% de pontualidade (Atlas Air), mas não para por ai, em Guarulhos a azul possui 89% vs latam com 75%.

Outra evidência é a marcação de algumas empresas que performam consistentemente mal, como por exemplo a TAP, com todas as performances inferiores a 71%. Assim, entre outros casos, podemos concluir que a empresa que possui uma operação não ideal e não o aeroporto, podendo intensificar a piora caso o aeroporto não tenha um histórico bom, porém boas empresas continuam com boas performances mesmo nesses aeroportos que possuem atrasos constantes.